# CLI 参数学习笔记

这份笔记专门介绍训练脚本和测试脚本的可传参数。

阅读目标：
- 知道主线 `train_ur5_reach.py` 可以传哪些参数
- 知道 Warp 训练线 `train_ur5_reach_warp.py` 可以传哪些参数
- 理解参数默认值、作用分组和常见组合方式

这份笔记和 `02_parameter_reference.ipynb` 的区别是：
- `02_parameter_reference.ipynb` 偏向配置层参数字典
- 当前笔记偏向命令行入口，也就是实际运行脚本时可以传入的参数


## 1. 主线 CLI 参数

主线入口文件是 `train_ur5_reach.py`。

这个脚本同时承担：
- 训练入口
- 测试入口
- 主线环境参数覆盖入口

也就是说，很多 `--xxx` 参数最终会映射到：
- `RLTrainConfig`
- `UR5ReachEnvConfig`


In [ ]:
import argparse
import pandas as pd

from train_ur5_reach import build_parser as build_main_parser
from train_ur5_reach_warp import build_parser as build_warp_parser


def parser_to_dataframe(parser: argparse.ArgumentParser) -> pd.DataFrame:
    rows = []
    for group in parser._action_groups:
        group_name = group.title
        for action in group._group_actions:
            if action.dest == 'help' or not action.option_strings:
                continue
            default = action.default
            if default is argparse.SUPPRESS:
                default = '<SUPPRESS>'
            choices = ''
            if action.choices is not None:
                choices = ', '.join(str(choice) for choice in action.choices)
            rows.append(
                {
                    'group': group_name,
                    'flags': ' / '.join(action.option_strings),
                    'dest': action.dest,
                    'default': default,
                    'choices': choices,
                    'help': action.help or '',
                }
            )
    return pd.DataFrame(rows)


main_parser = build_main_parser()
main_df = parser_to_dataframe(main_parser)
main_df.head()


## 2. 主线参数总表

下面这个表直接来自 `argparse`，因此它反映的是脚本真实支持的参数，而不是手工整理后的近似描述。

In [ ]:
main_df


## 3. 主线参数分组查看

主线脚本把参数分成几类：
- 基础运行参数
- `Training` 训练参数
- `Environment` 环境参数

这种分组方式有助于从“入口脚本怎么用”的角度理解项目。

In [ ]:
main_df.groupby('group')['dest'].count().to_frame('num_args')


In [ ]:
main_df[main_df['group'] == 'Training']


In [ ]:
main_df[main_df['group'] == 'Environment']


## 4. 主线常用参数组合

下面这些组合不是唯一方案，但很适合用来理解参数之间的配合关系。

### 4.1 最小训练命令

```bash
python train_ur5_reach.py \
  --algo sac \
  --run-name ur5_sac_main
```

重点参数：
- `--algo`：选择训练算法
- `--run-name`：决定产物目录名字


### 4.2 多并行训练

```bash
python train_ur5_reach.py \
  --algo sac \
  --run-name ur5_sac_parallel \
  --n-envs 4 \
  --total-timesteps 1500000
```

重点参数：
- `--n-envs`：控制并行环境数
- `--total-timesteps`：控制总训练步数


### 4.3 训练无头并行 + 主进程旁观

```bash
python train_ur5_reach.py \
  --algo sac \
  --run-name ur5_sac_watch \
  --n-envs 4 \
  --spectator-render \
  --spectator-render-every 200
```

重点参数：
- `--spectator-render`：打开独立旁观窗口
- `--spectator-render-every`：控制窗口更新频率


### 4.4 测试模式

```bash
python train_ur5_reach.py \
  --algo sac \
  --run-name ur5_sac_main \
  --test \
  --episodes 3 \
  --render
```

重点参数：
- `--test`：从训练入口切换到测试流程
- `--episodes`：测试回合数
- `--render`：是否打开 human 窗口


## 5. Warp CLI 参数

Warp 训练线入口文件是 `train_ur5_reach_warp.py`。

和主线相比，Warp 入口的重点更偏向：
- 大规模并行环境
- Brax 训练器参数
- JAX / Warp 兼容环境参数


In [ ]:
warp_parser = build_warp_parser()
warp_df = parser_to_dataframe(warp_parser)
warp_df.head()


## 6. Warp 参数总表

In [ ]:
warp_df


## 7. Warp 常用参数组合

### 7.1 Warp SAC 基础命令

```bash
python train_ur5_reach_warp.py \
  --algo sac \
  --run-name ur5_warp_sac \
  --num-envs 256 \
  --num-timesteps 5000000
```

重点参数：
- `--num-envs`：Warp 线吞吐的关键参数
- `--num-timesteps`：控制训练总步数


### 7.2 固定目标 + 位置增量控制

```bash
python train_ur5_reach_warp.py \
  --algo ppo \
  --run-name ur5_warp_fixed \
  --target-sampling-mode fixed \
  --controller-mode joint_position_delta \
  --reward-mode dense
```

重点参数：
- `--target-sampling-mode`：决定目标点采样方式
- `--controller-mode`：决定动作如何映射成控制量
- `--reward-mode`：决定奖励采用 dense 还是 sparse


## 8. 如何把 CLI 参数和代码实现对上

建议按下面顺序理解：

1. 先看 `build_parser()`，知道命令行支持哪些参数
2. 再看这些参数如何映射到 dataclass
3. 最后看环境类或训练函数里这些参数如何参与实现

主线建议对照文件：
- `train_ur5_reach.py`
- `ur5_reach_config.py`
- `ur5_reach_env.py`

Warp 建议对照文件：
- `train_ur5_reach_warp.py`
- `warp_ur5_config.py`
- `warp_ur5_env.py`
